# Public Library Resource Utilization Analysis

## Project overview
This notebook analyzes library visits and use of common public-library services across four branches over four months.

**Important:** The included data are synthetic, created for demonstration and coursework. They are not official records from any real library. Replace the CSV with approved, anonymized library data before drawing real-world conclusions.

### Objectives
- Compare visits and resource use across branches and months.
- Identify the most frequently used services.
- Calculate simple utilization indicators.
- Produce charts and a short, data-driven summary.

In [ ]:
# If needed, install dependencies in your environment:
# %pip install pandas matplotlib

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Put the CSV in the same folder as this notebook, or update this path.
CSV_PATH = Path("public_library_resource_utilization.csv")
df = pd.read_csv(CSV_PATH)
df.head()

In [ ]:
# Basic data checks
print("Rows:", len(df))
print("Columns:", list(df.columns))
print("\\nMissing values by column:")
print(df.isna().sum())
print("\\nDuplicate rows:", df.duplicated().sum())
df.describe(numeric_only=True).round(2)

In [ ]:
# Overall key performance indicators
service_cols = [
    "Books_Borrowed", "Computer_Sessions", "Reference_Queries",
    "Program_Attendees", "Study_Room_Uses"
]
summary = pd.Series({
    "Total visits": df["Visits"].sum(),
    "Total books borrowed": df["Books_Borrowed"].sum(),
    "Total computer sessions": df["Computer_Sessions"].sum(),
    "Total reference queries": df["Reference_Queries"].sum(),
    "Total program attendees": df["Program_Attendees"].sum(),
    "Total study-room uses": df["Study_Room_Uses"].sum(),
    "Books borrowed per visit": df["Books_Borrowed"].sum() / df["Visits"].sum()
}, name="Value")
summary.round(2)

In [ ]:
# Branch-level comparison
branch_summary = df.groupby("Library_Branch", as_index=False).agg(
    Visits=("Visits", "sum"),
    Books_Borrowed=("Books_Borrowed", "sum"),
    Computer_Sessions=("Computer_Sessions", "sum"),
    Reference_Queries=("Reference_Queries", "sum"),
    Program_Attendees=("Program_Attendees", "sum"),
    Study_Room_Uses=("Study_Room_Uses", "sum")
)
branch_summary.sort_values("Visits", ascending=False)

In [ ]:
# Monthly trends
month_order = ["Jan", "Feb", "Mar", "Apr"]
monthly = df.groupby("Month", as_index=False)[
    ["Visits", "Books_Borrowed", "Computer_Sessions"]
].sum()
monthly["Month"] = pd.Categorical(monthly["Month"], categories=month_order, ordered=True)
monthly = monthly.sort_values("Month")
monthly

In [ ]:
# Chart 1: visits by branch
visits_by_branch = df.groupby("Library_Branch")["Visits"].sum().sort_values(ascending=False)
ax = visits_by_branch.plot(kind="bar", figsize=(8, 4), title="Total Library Visits by Branch")
ax.set_xlabel("Library branch")
ax.set_ylabel("Visits")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: monthly trends in visits and borrowing
ax = monthly.set_index("Month")[["Visits", "Books_Borrowed"]].plot(
    kind="line", marker="o", figsize=(8, 4), title="Monthly Visits and Book Borrowing"
)
ax.set_xlabel("Month")
ax.set_ylabel("Recorded count")
plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: service-use totals
service_totals = df[service_cols].sum().sort_values(ascending=False)
ax = service_totals.plot(kind="bar", figsize=(9, 4), title="Total Recorded Use by Service")
ax.set_xlabel("Service")
ax.set_ylabel("Recorded count")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Simple utilization indicator: books borrowed per visit by branch
branch_rate = df.groupby("Library_Branch").apply(
    lambda g: g["Books_Borrowed"].sum() / g["Visits"].sum(),
    include_groups=False
).sort_values(ascending=False)
print("Books borrowed per visit by branch:")
display(branch_rate.rename("Books per visit").to_frame().round(3))

In [ ]:
# Automatically generated observations from the included sample
top_branch = branch_summary.loc[branch_summary["Visits"].idxmax(), "Library_Branch"]
top_month = monthly.loc[monthly["Visits"].idxmax(), "Month"]
service_totals = df[service_cols].sum().sort_values(ascending=False)

print(f"Highest-visit branch in this sample: {top_branch}")
print(f"Highest-visit month in this sample: {top_month}")
print(f"Most-used recorded service category: {service_totals.index[0]}")
print("\\nInterpret these as findings about the synthetic sample only, not real libraries.")

## Interpretation and recommendations
Use the outputs above to write findings based on the actual dataset you analyze. For a real study, consider:
1. Compare demand by branch and month before reallocating staff or materials.
2. Review computer and study-room use alongside the number of available devices/seats; counts alone do not measure capacity utilization.
3. Consider opening hours, local population, collection size, and special events when comparing branches.
4. Protect patron privacy: use aggregated counts and avoid personal borrowing histories or identifying information.

## Limitations
- The sample is synthetic and covers only four months and four branches.
- The dataset contains counts, not capacity, costs, satisfaction, or population served.
- Recorded service counts may overlap and should not automatically be added together as unique people or unique visits.

## Suggested conclusion
This project demonstrates how Python and pandas can summarize library activity, compare branches, examine monthly patterns, and visualize service use. Any operational decision should be based on verified local data and additional context.